# Fast Iterative Method (FIM) in CUDA

Implementazione del **Block FIM (Algoritmo 7)** della tesi su GPU, verificata e confrontata con il **Fast Marching (Algoritmo 3)** su CPU, come nella tesi (§4.1.2, speedup eq. 4.7).
Test: **sorgente puntiforme** in mezzo omogeneo (cap. 4.1), con stencil sugli assi e con **rotazione dello stencil** (§2.6.2).

**Prima di tutto:** *Runtime → Cambia tipo di runtime → T4 GPU*, poi esegui le celle in ordine.

## Controllo GPU
Da eseguire all'inizio di ogni sessione Colab: deve comparire una **Tesla T4**. Se non c'è: *Runtime → Cambia tipo di runtime → T4 GPU*.

In [ ]:
!nvidia-smi

## Riferimento CPU – `cpu_reference.h`

Il riferimento, come nella tesi, è il **Fast Marching sequenziale**: serve a **verificare** il risultato GPU e a misurare lo **speedup** (eq. 4.7).
- `solve_quadratic` → **Algoritmo 2** (eq. 3.1), `__host__ __device__`: la usano anche i kernel GPU;
- `solve_node` → *g(x)*: 4 vicini, oppure assi + stencil ruotato (`rotated = true`);
- `fmm_cpu` → **Algoritmo 3** (Fast Marching con min-heap);
- `relative_error` → eq. (4.2), confronto con U = √(x²+y²).

In [ ]:
%%writefile cpu_reference.h
// cpu_reference.h: funzioni condivise tra CPU e GPU + algoritmi di riferimento su CPU.
//   - solve_quadratic, solve_node : Algoritmo 2 (usate anche dai kernel GPU)
//   - fmm_cpu                     : Algoritmo 3, Fast Marching con min-heap
//   - relative_error              : eq. (4.2)
// Incluso da block_fim.cu. Il FMM e' il riferimento della tesi per lo speedup (eq. 4.7).
#pragma once
#include <cmath>
#include <vector>
#include <queue>
#include <functional>

// Valore "infinito": nodo non ancora raggiunto dal fronte d'onda.
// Usiamo un numero finito molto grande (non std::numeric_limits<float>::infinity())
// cosi' che a - b tra due INF faccia 0 e non NaN.
const float INF = 1e30f;

const float SQRT2 = 1.41421356f;

// Spostamenti (di, dj) verso i vicini.
// I primi 4 sono quelli degli assi (sinistra, destra, sotto, sopra);
// gli ultimi 4 sono i diagonali, usati solo con la rotazione dello stencil.
const int DI[8] = {  0, 0, -1, 1,   -1, -1,  1, 1 };
const int DJ[8] = { -1, 1,  0, 0,   -1,  1, -1, 1 };

// ---------------------------------------------------------------------------
// Algoritmo 2 della tesi: soluzione dell'equazione quadratica (3.1)
//   (u - a)^2 + (u - b)^2 = (F h)^2
// a = minimo dei vicini lungo x, b = minimo dei vicini lungo y.
// __host__ __device__: la stessa funzione servira' anche nei kernel GPU.
// ---------------------------------------------------------------------------
__host__ __device__ float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)      // il fronte arriva da entrambe le direzioni
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;     // il fronte arriva da una sola direzione
}

// ---------------------------------------------------------------------------
// g(x) della tesi: calcola il nuovo valore nel nodo (i, j).
// Legge i 4 vicini (INF se fuori dal dominio) e chiama l'Algoritmo 2.
// Con rotated = true applica anche la rotazione dello stencil (tesi, §2.6.2):
// lo stesso Algoritmo 2 sui vicini diagonali con passo sqrt(2) h, e si tiene il minimo.
// Restituisce min(u_vecchio, u_nuovo): u puo' solo diminuire (eq. 3.2).
// Riceve puntatori (non std::vector) ed e' __host__ __device__:
// la CPU la chiama con u.data(), i kernel GPU con i puntatori alla memoria della GPU.
// ---------------------------------------------------------------------------
__host__ __device__ float solve_node(const float* u, const float* F,
                                     int i, int j, int N, float h, bool rotated)
{
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);           // stencil sugli assi

    if (rotated) {
        // Stencil ruotato di pi/4: i "nuovi assi" sono le due diagonali.
        float down_left  = (i > 0 && j > 0)         ? u[(i - 1) * N + (j - 1)] : INF;
        float up_right   = (i < N - 1 && j < N - 1) ? u[(i + 1) * N + (j + 1)] : INF;
        float up_left    = (i < N - 1 && j > 0)     ? u[(i + 1) * N + (j - 1)] : INF;
        float down_right = (i > 0 && j < N - 1)     ? u[(i - 1) * N + (j + 1)] : INF;

        float a_rot = fminf(down_left, up_right);         // minimo lungo la diagonale "in salita a destra"
        float b_rot = fminf(up_left, down_right);         // minimo lungo la diagonale "in salita a sinistra"
        float q_rot = solve_quadratic(a_rot, b_rot, F[idx], SQRT2 * h);

        q = fminf(q, q_rot);                              // causalita': vince il minore
    }
    return fminf(u[idx], q);
}

// ---------------------------------------------------------------------------
// Algoritmo 3 (Fast Marching), versione sequenziale.
//
// Stato dei nodi:  LONTANO (non raggiunto), ATTIVO (nella Narrow Band), NOTO (congelato).
// A ogni passo si estrae dalla Narrow Band il nodo con u minima (Trial): per
// causalita' nessun altro nodo puo' piu' migliorarlo, quindi diventa NOTO.
// Poi si calcola g(x) per i suoi vicini non NOTI, che entrano nella Narrow Band.
//
// La Narrow Band e' un min-heap (std::priority_queue con std::greater):
// top() restituisce la coppia (u, indice) con u minima.
// std::priority_queue non permette di modificare un elemento gia' inserito:
// quando u di un nodo diminuisce si inserisce una NUOVA coppia, e le coppie
// vecchie ("scadute") si scartano quando arrivano in cima.
// ---------------------------------------------------------------------------
void fmm_cpu(std::vector<float>& u, const std::vector<float>& F,
             int N, float h, int src, bool rotated)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };
    std::vector<char> state(N * N, LONTANO);

    typedef std::pair<float, int> Entry;                       // (u, indice del nodo)
    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;

    heap.push(Entry(u[src], src));                             // la sorgente e' il primo Trial
    state[src] = ATTIVO;
    int n_nb = rotated ? 8 : 4;

    while (!heap.empty()) {
        Entry e = heap.top();                                  // Trial <- radice del min-heap
        heap.pop();                                            // rimuovi Trial da A
        int x = e.second;
        if (state[x] == NOTO || e.first > u[x]) continue;      // coppia scaduta: ignorala
        state[x] = NOTO;                                       // aggiungi Trial ai Noti

        int i = x / N, j = x % N;
        for (int k = 0; k < n_nb; ++k) {                       // foreach x_nb adiacente a Trial
            int ni = i + DI[k], nj = j + DJ[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;
            int nb = ni * N + nj;
            if (state[nb] == NOTO) continue;                   // if x_nb non e' Noto

            float q = solve_node(u.data(), F.data(), ni, nj, N, h, rotated);
            if (q < u[nb]) {                                   // u(x_nb) <- g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                            // aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

// ---------------------------------------------------------------------------
// Errore relativo, eq. (4.2): ||u - U|| / ||U||, con U = sqrt(x^2 + y^2).
// ---------------------------------------------------------------------------
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h;
            double y = -1.0 + i * h;
            double U = std::sqrt(x * x + y * y);
            double d = u[i * N + j] - U;
            num += d * d;
            den += U * U;
        }
    return std::sqrt(num / den);
}

## FIM – Block Fast Iterative Method (Algoritmo 7) su GPU

La griglia N×N è divisa in **blocchi di 8×8 nodi**; la lista A contiene **blocchi**, e ogni blocco attivo è elaborato da un **blocco CUDA di 8×8 thread** (un thread per nodo).

| Algoritmo 7 | `block_fim.cu` |
|---|---|
| Step 1: foreach b ∈ A: `niter` iterazioni locali, Cb(b) = reduction(Cn(b)) | `update_active_tiles` → `process_tile(niter)` |
| Step 2: vicini dei blocchi convergenti: 1 iterazione + reduction | `update_neighbor_tiles` (*gather*) → `process_tile(1)` |
| Step 3: A = { b : Cb(b) = falso } | `build_active_list` (`atomicAdd`) |

`process_tile`: tile + **halo** in **shared memory** → `niter` iterazioni con `__syncthreads()` → scrittura in memoria globale → **reduction AND** (fig. 3.7).

L'output confronta la GPU con il **Fast Marching**: max|GPU−FMM| (correttezza) e speedup t_FMM / t_FIM (eq. 4.7).

In [ ]:
%%writefile block_fim.cu
// block_fim.cu: Block FIM (Algoritmo 7 della tesi) su GPU.
// Il risultato viene confrontato nodo per nodo con il Fast Marching su CPU (fmm_cpu),
// che e' anche il riferimento per lo speedup (eq. 4.7 della tesi).
#include <cstdio>
#include <cstdlib>
#include <chrono>
#include <fstream>
#include <cuda_runtime.h>
#include "cpu_reference.h"   // solve_quadratic (__host__ __device__), fmm_cpu, relative_error

#define CUDA_CHECK(call)                                                    \
    do {                                                                    \
        cudaError_t err = (call);                                           \
        if (err != cudaSuccess) {                                           \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,           \
                   cudaGetErrorString(err));                                \
            exit(1);                                                        \
        }                                                                   \
    } while (0)

// Un blocco b della tesi = TILE x TILE nodi = un blocco CUDA di TILE x TILE thread.
// E' una costante di compilazione perche' dimensiona la shared memory.
// TILE * TILE deve essere una potenza di 2 per la reduction (fig. 3.7).
#define TILE 8

// ===========================================================================
// Algoritmo 2 applicato a un nodo della tile in shared memory.
//
// La tile ha un bordo (halo) di 1 nodo su ogni lato, copiato dai blocchi vicini:
//
//      s[0][0]  s[0][1] ......... s[0][TILE]   s[0][TILE+1]      <- halo
//      s[1][0]  | nodi del blocco: s[1..TILE][1..TILE] |  s[1][TILE+1]
//        ...    |                                      |    ...
//      s[TILE+1][0] ......................... s[TILE+1][TILE+1] <- halo
//
// Il nodo (ty, tx) del blocco sta in s[ty + 1][tx + 1].
// I nodi fuori dal dominio sono stati caricati come INF: niente controlli sui bordi.
// Stessa formula di solve_node (cpu_reference.h), ma i vicini si leggono dalla shared memory.
// ===========================================================================
__device__ float solve_in_tile(const float s[TILE + 2][TILE + 2], int y, int x,
                               float F, float h, bool rotated)
{
    float a = fminf(s[y][x - 1], s[y][x + 1]);        // sinistra, destra  (asse x)
    float b = fminf(s[y - 1][x], s[y + 1][x]);        // sotto, sopra      (asse y)
    float q = solve_quadratic(a, b, F, h);

    if (rotated) {                                    // stencil ruotato (§2.6.2)
        float a_rot = fminf(s[y - 1][x - 1], s[y + 1][x + 1]);
        float b_rot = fminf(s[y + 1][x - 1], s[y - 1][x + 1]);
        q = fminf(q, solve_quadratic(a_rot, b_rot, F, SQRT2 * h));
    }
    return fminf(s[y][x], q);                         // u puo' solo diminuire
}

// ===========================================================================
// Aggiornamento di UN blocco (tile) da parte di un blocco CUDA:
//
//   for i = 0 to niter:  (U(b), Cn(b)) = soluzione di g(b)
//   Cb(b) = reduction(Cn(b))
//
// 1. copia tile + halo dalla memoria globale alla shared memory
// 2. niter iterazioni locali, tutte in shared memory
// 3. copia la tile aggiornata nella memoria globale
// 4. reduction AND dei flag Cn -> Cb(b), scritto in Cb_out[tile]
//
// Usata sia dallo Step 1 (niter iterazioni) sia dallo Step 2 (1 iterazione).
// Deve essere chiamata da TUTTI i thread del blocco (contiene __syncthreads).
// ===========================================================================
__device__ void process_tile(int tile, int niter, float* u, const float* F,
                             int N, int nbx, float h, float eps, int src,
                             bool rotated, unsigned char* Cb_out)
{
    __shared__ float         s_u[TILE + 2][TILE + 2];   // tile + halo
    __shared__ unsigned char s_c[TILE * TILE];          // flag Cn per la reduction

    int tx = threadIdx.x, ty = threadIdx.y;
    int tid = ty * TILE + tx;                  // indice del thread nel blocco: 0 .. TILE*TILE-1

    int bi = tile / nbx, bj = tile % nbx;      // riga e colonna del blocco nella griglia di blocchi
    int i = bi * TILE + ty;                    // riga    globale del nodo di questo thread
    int j = bj * TILE + tx;                    // colonna globale del nodo di questo thread
    bool inside = (i < N && j < N);            // l'ultima riga/colonna di blocchi puo' sporgere
    int idx = i * N + j;
    bool fixed = !inside || idx == src;        // nodi da non aggiornare: fuori dominio o sorgente

    // ---- 1. memoria globale -> shared (tile + halo) ----
    // (TILE+2)^2 = 100 celle e TILE^2 = 64 thread: ogni thread copia una cella,
    // poi salta avanti di 64 finche' non sono finite. Copre anche gli angoli dell'halo.
    for (int k = tid; k < (TILE + 2) * (TILE + 2); k += TILE * TILE) {
        int sy = k / (TILE + 2), sx = k % (TILE + 2);
        int gi = bi * TILE + sy - 1;           // -1: la cella s[0][*] e' la riga sopra al blocco
        int gj = bj * TILE + sx - 1;
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N);
        s_u[sy][sx] = in_domain ? u[gi * N + gj] : INF;
    }
    float f = inside ? F[idx] : 1.0f;
    __syncthreads();                           // la tile e' completa prima di usarla

    // ---- 2. niter iterazioni locali in shared memory ----
    bool conv = true;
    for (int it = 0; it < niter; ++it) {
        float p = s_u[ty + 1][tx + 1];
        float q = fixed ? p : solve_in_tile(s_u, ty + 1, tx + 1, f, h, rotated);
        __syncthreads();                       // tutti hanno letto i vicini...
        s_u[ty + 1][tx + 1] = q;
        __syncthreads();                       // ...prima che qualcuno li sovrascriva
        conv = fabsf(p - q) < eps;             // Cn del nodo all'ultima iterazione
    }

    // ---- 3. shared -> memoria globale (solo i nodi del blocco, non l'halo) ----
    if (inside) u[idx] = s_u[ty + 1][tx + 1];

    // ---- 4. reduction AND: Cb(b) = vero solo se tutti i nodi sono convergenti ----
    // Come in fig. 3.7: a ogni passo i primi 'stride' thread fanno l'AND tra
    // l'elemento tid e l'elemento tid + stride; l'array si dimezza.
    // 64 -> 32 -> 16 -> 8 -> 4 -> 2 -> 1: log2(64) = 6 passi.
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = TILE * TILE / 2; stride > 0; stride /= 2) {
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[tile] = s_c[0];
}

// ===========================================================================
// KERNEL Step 1 dell'Algoritmo 7: aggiornamento dei blocchi attivi.
//   foreach b in A in parallelo: niter iterazioni locali + reduction
// Si lanciano SOLO i blocchi attivi: il blocco CUDA k elabora il blocco list[k].
// Il risultato (convergente si'/no) va in Cb1.
// ===========================================================================
__global__ void update_active_tiles(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, float eps, int src,
                                    bool rotated, int niter, unsigned char* Cb1)
{
    process_tile(list[blockIdx.x], niter, u, F, N, nbx, h, eps, src, rotated, Cb1);
}

// ===========================================================================
// KERNEL Step 2 dell'Algoritmo 7: aggiornamento dei blocchi vicini.
//   foreach b in A: if Cb(b) then foreach b_nb adiacente: (U, Cn) = g(b_nb); Cb(b_nb) = reduction
//
// Come nello Step 2 del FIM per nodi usiamo il "gather": ogni blocco CUDA guarda
// il PROPRIO blocco e si chiede "un mio blocco vicino e' appena converso?".
// Cosi' nessun blocco viene elaborato da due blocchi CUDA nello stesso momento.
//
// Come nell'Algoritmo 6 ("if x_nb non in A"), un blocco ancora attivo e non
// convergente NON viene testato: resta in A e verra' rielaborato al giro dopo.
// Il risultato finale (Cb) di ogni blocco va in Cb.
// ===========================================================================
__global__ void update_neighbor_tiles(float* u, const float* F,
                                      int N, int nbx, float h, float eps, int src, bool rotated,
                                      const unsigned char* active, const unsigned char* Cb1,
                                      unsigned char* Cb)
{
    int bi = blockIdx.y, bj = blockIdx.x;      // un blocco CUDA per ogni blocco della griglia
    int tile = bi * nbx + bj;
    bool first = (threadIdx.x == 0 && threadIdx.y == 0);   // un solo thread scrive Cb

    // Caso 1: blocco in A e non convergente -> resta in A, nessun test.
    if (active[tile] && !Cb1[tile]) {
        if (first) Cb[tile] = 0;
        return;
    }

    // Caso 2: blocco fuori da A, o appena converso nello Step 1.
    // C'e' un blocco vicino "appena converso" (era in A e lo Step 1 ha dato Cb = vero)?
    // Con la rotazione un nodo d'angolo dipende dal blocco in diagonale: 8 vicini.
    const int di[8] = {  0, 0, -1, 1,   -1, -1,  1, 1 };
    const int dj[8] = { -1, 1,  0, 0,   -1,  1, -1, 1 };
    int n_nb = rotated ? 8 : 4;
    bool near_converged = false;
    for (int k = 0; k < n_nb; ++k) {
        int ni = bi + di[k], nj = bj + dj[k];
        if (ni < 0 || ni >= nbx || nj < 0 || nj >= nbx) continue;
        int t = ni * nbx + nj;
        if (active[t] && Cb1[t]) { near_converged = true; break; }
    }

    // Tutti i thread del blocco hanno calcolato le stesse condizioni:
    // o escono tutti o restano tutti (necessario per i __syncthreads di process_tile).
    if (!near_converged) {
        if (first) Cb[tile] = 1;               // nessuna novita' dai vicini: convergente
        return;
    }
    process_tile(tile, 1, u, F, N, nbx, h, eps, src, rotated, Cb);   // test: 1 iterazione
}

// ===========================================================================
// KERNEL Step 3 dell'Algoritmo 7: nuova lista dei blocchi attivi.
//   clear(A); foreach b: if Cb(b) = FALSO then aggiungi b ad A
// Un thread per blocco della griglia. La posizione in lista si ottiene con
// atomicAdd: incrementa *count e restituisce il valore PRIMA dell'incremento,
// in modo indivisibile -> ogni blocco attivo ottiene una posizione diversa.
// ===========================================================================
__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_tiles)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= n_tiles) return;
    bool act = !Cb[t];
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;
}

// ===========================================================================
// Ciclo dell'Algoritmo 7 sull'host.
// Restituisce il numero di iterazioni globali; in *ms il tempo GPU.
// ===========================================================================
int block_fim_gpu(std::vector<float>& u, const std::vector<float>& F,
                  int N, float h, int src, float eps, bool rotated, int niter,
                  float* ms)
{
    const size_t n = (size_t)N * N;
    const int nbx = (N + TILE - 1) / TILE;     // blocchi per lato (l'ultimo puo' sporgere)
    const int n_tiles = nbx * nbx;

    // ---- 1. Memoria GPU ----
    float *d_u, *d_F;
    unsigned char *d_active, *d_Cb1, *d_Cb;
    int *d_list, *d_count;
    CUDA_CHECK(cudaMalloc(&d_u,      n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_F,      n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_active, n_tiles));
    CUDA_CHECK(cudaMalloc(&d_Cb1,    n_tiles));
    CUDA_CHECK(cudaMalloc(&d_Cb,     n_tiles));
    CUDA_CHECK(cudaMalloc(&d_list,   n_tiles * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count,  sizeof(int)));

    // ---- 2. Stato iniziale ----
    // Tutti i blocchi "convergenti" (Cb = 1) tranne quello che contiene la sorgente:
    // il primo build_active_list mettera' in A solo quel blocco.
    std::vector<unsigned char> Cb0(n_tiles, 1);
    int src_tile = ((src / N) / TILE) * nbx + (src % N) / TILE;
    Cb0[src_tile] = 0;
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), n * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), n * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_tiles, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_tiles));

    dim3 tile_threads(TILE, TILE);             // un thread per nodo del blocco
    dim3 all_tiles(nbx, nbx);                  // un blocco CUDA per ogni blocco della griglia
    int list_threads = 256;
    int list_blocks  = (n_tiles + list_threads - 1) / list_threads;

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));

    // ---- 3. Ciclo principale: while (A non vuota) ----
    int iter = 0;
    while (true) {
        // Step 3 (fatto per primo): costruisce A dai Cb correnti
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, list_threads>>>(d_Cb, d_active, d_list, d_count, n_tiles);
        CUDA_CHECK(cudaGetLastError());

        int n_active = 0;
        CUDA_CHECK(cudaMemcpy(&n_active, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;              // A vuota: fine
        ++iter;

        // Step 1: solo i blocchi attivi -> n_active blocchi CUDA
        update_active_tiles<<<n_active, tile_threads>>>(d_list, d_u, d_F, N, nbx, h, eps, src,
                                                        rotated, niter, d_Cb1);
        CUDA_CHECK(cudaGetLastError());

        // Step 2: blocchi vicini a quelli appena convergenti
        update_neighbor_tiles<<<all_tiles, tile_threads>>>(d_u, d_F, N, nbx, h, eps, src, rotated,
                                                           d_active, d_Cb1, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }

    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));

    // ---- 4. Risultato e pulizia ----
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, n * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_u));
    CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_active));
    CUDA_CHECK(cudaFree(d_Cb1));
    CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_list));
    CUDA_CHECK(cudaFree(d_count));
    return iter;
}

// ===========================================================================
// Confronto Block FIM (GPU) con Fast Marching (CPU).
// Per N = 101 salva la soluzione GPU su file per il grafico.
// ===========================================================================
void run_test(bool rotated, int niter, const char* filename)
{
    const float eps = 1e-6f;
    const int sizes[] = { 101, 401, 1001, 2001, 4001 };

    printf("\n===== %s, TILE = %d, niter = %d =====\n",
           rotated ? "Assi + ruotato (8 vicini)" : "Stencil assi (4 vicini)", TILE, niter);
    printf("   N  iter glob.  max|GPU-FMM|  errore rel.  t FMM CPU [ms]  t FIM GPU [ms]  speedup\n");
    for (int N : sizes) {
        float h   = 2.0f / (N - 1);
        int   src = (N / 2) * N + (N / 2);
        std::vector<float> F(N * N, 1.0f);

        // Fast Marching su CPU (Algoritmo 3): riferimento della tesi
        std::vector<float> u_fmm(N * N, INF);
        u_fmm[src] = 0.0f;
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm_cpu(u_fmm, F, N, h, src, rotated);
        auto t1 = std::chrono::high_resolution_clock::now();
        double ms_fmm = std::chrono::duration<double, std::milli>(t1 - t0).count();

        // Block FIM su GPU (Algoritmo 7)
        std::vector<float> u_gpu(N * N, INF);
        u_gpu[src] = 0.0f;
        float ms_gpu = 0.0f;
        int iter = block_fim_gpu(u_gpu, F, N, h, src, eps, rotated, niter, &ms_gpu);

        float max_diff = 0.0f;
        for (int k = 0; k < N * N; ++k) max_diff = fmaxf(max_diff, fabsf(u_gpu[k] - u_fmm[k]));

        printf("%5d  %9d     %.2e    %.3e  %14.2f  %14.2f  %7.2f\n",
               N, iter, max_diff, relative_error(u_gpu, N, h), ms_fmm, ms_gpu, ms_fmm / ms_gpu);

        if (N == 101) {
            std::ofstream out(filename, std::ios::binary);
            out.write(reinterpret_cast<const char*>(u_gpu.data()), N * N * sizeof(float));
        }
    }
}

int main()
{
    CUDA_CHECK(cudaFree(0));                   // inizializza la GPU fuori dalle misure

    const int niter = 2 * TILE;                // iterazioni locali per blocco
    run_test(false, niter, "block_fim_assi.bin");
    run_test(true,  niter, "block_fim_rot.bin");
    printf("\nSoluzioni GPU N = 101 salvate in block_fim_assi.bin e block_fim_rot.bin\n");
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 block_fim.cu -o block_fim && ./block_fim

### Visualizzazione (N = 101): stencil sugli assi vs stencil ruotato
In alto: curve di livello del Block FIM (continue) sovrapposte a quelle analitiche (tratteggiate), come in fig. 4.1.
In basso: errore |u − U| in ogni nodo, con la **stessa scala di colori**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 101
x = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, x)
U = np.sqrt(X**2 + Y**2)                                         # soluzione analitica (eq. 4.3)

casi = [('block_fim_assi.bin', 'Stencil assi (4 vicini)'),
        ('block_fim_rot.bin',  'Assi + ruotato (8 vicini)')]
sol = [np.fromfile(f, dtype=np.float32).reshape((N, N)) for f, _ in casi]   # riga i = asse y, colonna j = asse x
err_max = max(np.abs(u - U).max() for u in sol)

fig, ax = plt.subplots(2, 2, figsize=(12, 11))
levels = np.arange(0.2, 1.5, 0.2)

for c, (u, (_, titolo)) in enumerate(zip(sol, casi)):
    c1 = ax[0, c].contour(X, Y, u, levels=levels, colors='tab:blue')
    ax[0, c].contour(X, Y, U, levels=levels, colors='k', linestyles='--')
    ax[0, c].clabel(c1, fmt='%.1f', fontsize=8)
    ax[0, c].plot(0, 0, 'ro')
    ax[0, c].set_aspect('equal')
    ax[0, c].set_title(titolo + '\nFIM (blu) vs analitica (tratteggio)')

    im = ax[1, c].imshow(np.abs(u - U), origin='lower', extent=[-1, 1, -1, 1],
                         cmap='viridis', vmin=0, vmax=err_max)
    ax[1, c].set_title(titolo + '\nerrore |u − U|')
    fig.colorbar(im, ax=ax[1, c], shrink=0.8)

for a in ax.flat:
    a.set_xlabel('x [u.a.]'); a.set_ylabel('y [u.a.]')

plt.tight_layout()
plt.show()